## Model Compression baseline

The purpose of this notebook is to define baseline compression strategies. These strategies will be compared to advanced personalized compression strategies in the thesis to determine their usefulness.

setup

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import torch

In [2]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

In [3]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlp_replacement.config import (
    CaptureConfig,
    DataConfig,
    ExperimentConfig,
    ModelConfig,
    OperatorConfig,
    RecoveryConfig,
    SelectionConfig,
    WorkflowConfig,
)
from mlp_replacement.data import build_data_loaders
from mlp_replacement.evaluation.footprint import (
    parameter_footprint,
    serialized_checkpoint_bytes,
)
from mlp_replacement.evaluation.language_model import (
    evaluate_language_model,
)
from mlp_replacement.model import (
    discover_mlp_blocks,
    load_model_and_tokenizer,
)
from mlp_replacement.runlog import environment_record, json_value
from mlp_replacement.compression.selection import select_layers
from mlp_replacement.compression.workflows import run_replacement_experiment

In [4]:
SEED = 21
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

In [5]:
EXECUTION_MODE = 'run'
RUN_FROM_SCRATCH = EXECUTION_MODE == 'run'
loaded_artifact = None

SWIGLU_WIDTH_RATIO = 0.50
INTERLEAVE_STRIDE = 2
INTERLEAVE_OFFSET = 0
RECOVERY_BATCHES = 64
SAVE_COMPRESSED_MODEL = False
COMPRESSED_MODEL_ROOT = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'compressed-models'
    / 'interleaved-swiglu-050'
)
COMPRESSED_MODEL_PATHS = {
    'including_boundaries': (
        COMPRESSED_MODEL_ROOT / 'including-boundaries'
    ),
    'excluding_boundaries': (
        COMPRESSED_MODEL_ROOT / 'excluding-boundaries'
    ),
}
ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'model-compression-baselines'
    / 'compression-baseline.json'
)

In [6]:
model_config = ModelConfig(
    model_id='HuggingFaceTB/SmolLM2-1.7B',
    device='auto',
    dtype='auto',
)

data_config = DataConfig(
    sequence_length=128,
    batch_size=2,
    num_calibration_batches=48,
    num_operator_validation_batches=24,
    num_recovery_batches=RECOVERY_BATCHES,
    num_recovery_validation_batches=24,
    num_model_validation_batches=24,
    num_test_batches=0,
    seed=SEED,
)

selection_including_boundaries = SelectionConfig(
    strategy='interleaved',
    interleave_stride=INTERLEAVE_STRIDE,
    interleave_offset=INTERLEAVE_OFFSET,
    protected_prefix=0,
    protected_suffix=0,
    application_order='layer',
    seed=SEED,
)

selection_excluding_boundaries = SelectionConfig(
    strategy='interleaved',
    interleave_stride=INTERLEAVE_STRIDE,
    interleave_offset=INTERLEAVE_OFFSET,
    protected_prefix=1,
    protected_suffix=1,
    application_order='layer',
    seed=SEED,
)

operator_config = OperatorConfig(
    kind='swiglu',
    intermediate_ratio=SWIGLU_WIDTH_RATIO,
    bias=False,
    epochs=64,
    learning_rate=1e-3,
    batch_size=2048,
    weight_decay=0.0,
    scheduler='constant',
    early_stopping_patience=3,
    seed=SEED,
)

recovery_config = RecoveryConfig(
    enabled=True,
    epochs=1,
    learning_rate=1e-5,
    weight_decay=0.0,
    temperature=1.0,
    cache_dtype='float16',
    early_stopping_patience=None,
)

capture_config = CaptureConfig(
    storage_device='cpu',
    storage_dtype='float32',
)
workflow_config = WorkflowConfig(strategy='one_shot')

experiment_configs = {
    'including_boundaries': ExperimentConfig(
        model=model_config,
        data=data_config,
        capture=capture_config,
        selection=selection_including_boundaries,
        operator=operator_config,
        recovery=recovery_config,
        workflow=workflow_config,
    ),
    'excluding_boundaries': ExperimentConfig(
        model=model_config,
        data=data_config,
        capture=capture_config,
        selection=selection_excluding_boundaries,
        operator=operator_config,
        recovery=recovery_config,
        workflow=workflow_config,
    ),
}

In [7]:
if RUN_FROM_SCRATCH:
    model, tokenizer = load_model_and_tokenizer(model_config)
    device = next(model.parameters()).device
else:
    loaded_artifact = json.loads(
        ARTIFACT_PATH.read_text(encoding='utf-8')
    )

/home/sgulas/miniconda3/envs/mlp-replacement/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 218/218 [00:00<00:00, 2369.71it/s]


In [8]:
if RUN_FROM_SCRATCH:
    loaders = build_data_loaders(
        tokenizer,
        data_config,
        include_recovery=True,
    )

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (268279 > 8192). Running this sequence through the model will result in indexing errors


### Initial evaluation

In [9]:
if RUN_FROM_SCRATCH:
    mlp_blocks = discover_mlp_blocks(model)
    available_layers = [block.index for block in mlp_blocks]
    selection_previews = {
        name: select_layers(
            available_layers,
            config.selection,
        )
        for name, config in experiment_configs.items()
    }
    initial_footprint = parameter_footprint(model)
    initial_metrics = evaluate_language_model(
        model,
        loaders.model_validation,
        device,
        data_config.num_model_validation_batches,
    )

    initial_evaluation = pd.DataFrame([{
        'variant': 'dense_reference',
        'stage': 'initial',
        'selected_layers': [],
        'parameters': initial_footprint.parameters,
        'trainable_parameters': (
            initial_footprint.trainable_parameters
        ),
        'theoretical_weight_bytes': (
            initial_footprint.theoretical_weight_bytes
        ),
        'removed_parameters': 0,
        'parameter_reduction_pct': 0.0,
        'teacher_kl': 0.0,
        'loss': initial_metrics.loss,
        'perplexity': initial_metrics.perplexity,
        'predicted_tokens': initial_metrics.predicted_tokens,
        'batches': initial_metrics.batches,
    }])
    selection_summary = pd.DataFrame([
        {
            'variant': name,
            'eligible_blocks': len(selection.eligible_indices),
            'selected_blocks': len(selection.indices),
            'selected_layers': list(selection.indices),
        }
        for name, selection in selection_previews.items()
    ])
else:
    initial_evaluation = pd.DataFrame(
        loaded_artifact['results']['evaluations']
    ).query("variant == 'dense_reference'").reset_index(drop=True)
    selection_summary = pd.DataFrame(
        loaded_artifact['results']['selections']
    )

display(initial_evaluation)
display(selection_summary)

,variant,stage,selected_layers,parameters,trainable_parameters,theoretical_weight_bytes,removed_parameters,parameter_reduction_pct,teacher_kl,loss,perplexity,predicted_tokens,batches
0,dense_reference,initial,[],1711376384,1711376384,3422752768,0,0.0,0.0,2.669973,14.439586,6096,24


,variant,eligible_blocks,selected_blocks,selected_layers
0,including_boundaries,24,12,"[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]"
1,excluding_boundaries,22,11,"[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]"


## Experiments

### Uniform Compression baselines

1. Interleaved swiGLU replacement:
    - replace every second block of the model with selected operator (e.g. swiGLU 0.50)
    - KD retrain

2. BI-score guided k-block replacement:
    - extension of mvp
    - test BI/MLP-score
    - approach:
        1. select target model % sparsity = s
        2. redistribute uniform operator capacity based on s -> number of replaced blocks k
        3. replace top/bottom k blocks
        4. KD retrain


constraints:
- exclusion of transformer first and last block

#### Interleaved swiGLU

In [10]:
interleaved_results = {}
serialized_model_bytes_by_variant = {}

for run_index, (variant, config) in enumerate(
    experiment_configs.items() if RUN_FROM_SCRATCH else []
):
    if run_index > 0:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model, _ = load_model_and_tokenizer(model_config)

    interleaved_results[variant] = run_replacement_experiment(
        model,
        loaders,
        config,
    )

    serialized_model_bytes_by_variant[variant] = None
    if SAVE_COMPRESSED_MODEL:
        output_path = COMPRESSED_MODEL_PATHS[variant]
        model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        serialized_model_bytes_by_variant[variant] = (
            serialized_checkpoint_bytes(output_path)
        )

Loading weights: 100%|██████████| 218/218 [00:00<00:00, 4800.92it/s]


In [11]:
compressed_rows = []
block_rows = []

for variant, result in interleaved_results.items():
    removed_parameters = (
        result.footprint_before.parameters
        - result.footprint_after.parameters
    )
    parameter_reduction_pct = (
        100
        * removed_parameters
        / result.footprint_before.parameters
    )
    stages = (
        (
            'before_recovery',
            result.pre_recovery_validation_metrics,
            result.pre_recovery_validation_kl,
        ),
        (
            'after_recovery',
            result.final_validation_metrics,
            result.post_recovery_validation_kl,
        ),
    )

    for stage, metrics, teacher_kl in stages:
        compressed_rows.append({
            'variant': variant,
            'stage': stage,
            'selected_layers': list(result.selection.indices),
            'parameters': result.footprint_after.parameters,
            'trainable_parameters': (
                result.footprint_after.trainable_parameters
            ),
            'theoretical_weight_bytes': (
                result.footprint_after.theoretical_weight_bytes
            ),
            'removed_parameters': removed_parameters,
            'parameter_reduction_pct': parameter_reduction_pct,
            'teacher_kl': teacher_kl,
            'loss': metrics.loss,
            'perplexity': metrics.perplexity,
            'predicted_tokens': metrics.predicted_tokens,
            'batches': metrics.batches,
        })

    for block in result.blocks:
        block_rows.append({
            'variant': variant,
            'layer': block.layer_index,
            'original_parameters': block.original_parameters,
            'replacement_parameters': block.replacement_parameters,
            'retained_pct': (
                100
                * block.replacement_parameters
                / block.original_parameters
            ),
            'validation_mse': block.operator_validation_mse,
            'best_epoch': block.best_operator_epoch,
        })

if RUN_FROM_SCRATCH:
    compressed_evaluation = pd.DataFrame(
        compressed_rows,
        columns=initial_evaluation.columns,
    )
    evaluation_summary = pd.concat(
        [initial_evaluation, compressed_evaluation],
        ignore_index=True,
    )
    block_summary = pd.DataFrame(block_rows)
else:
    evaluation_summary = pd.DataFrame(
        loaded_artifact['results']['evaluations']
    )
    compressed_evaluation = evaluation_summary.query(
        "variant != 'dense_reference'"
    ).reset_index(drop=True)
    block_summary = pd.DataFrame(
        loaded_artifact['results']['block_fitting']
    )

display(evaluation_summary)
display(block_summary)

,variant,stage,selected_layers,parameters,trainable_parameters,theoretical_weight_bytes,removed_parameters,parameter_reduction_pct,teacher_kl,loss,perplexity,predicted_tokens,batches
0,dense_reference,initial,[],1711376384,1711376384,3422752768,0,0.000000,0.000000,2.669973,1.443959e+01,6096,24
1,including_boundaries,before_recovery,"[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]",1409386496,1409386496,2818772992,301989888,17.646024,14.627843,17.654531,4.647990e+07,6096,24
2,including_boundaries,after_recovery,"[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]",1409386496,1409386496,2818772992,301989888,17.646024,13.319576,14.827358,2.750678e+06,6096,24
3,excluding_boundaries,before_recovery,"[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]",1434552320,1434552320,2869104640,276824064,16.175522,2.966435,6.190840,4.882558e+02,6096,24
4,excluding_boundaries,after_recovery,"[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21]",1434552320,1434552320,2869104640,276824064,16.175522,1.674140,5.204122,1.820211e+02,6096,24


,variant,layer,original_parameters,replacement_parameters,retained_pct,validation_mse,best_epoch
0,including_boundaries,0,50331648,25165824,50.0,0.051245,64
1,including_boundaries,2,50331648,25165824,50.0,0.152653,44
2,including_boundaries,4,50331648,25165824,50.0,0.307208,38
3,including_boundaries,6,50331648,25165824,50.0,0.493018,35
4,including_boundaries,8,50331648,25165824,50.0,0.876719,41
5,including_boundaries,10,50331648,25165824,50.0,1.004025,45
6,including_boundaries,12,50331648,25165824,50.0,1.064269,44
7,including_boundaries,14,50331648,25165824,50.0,1.571612,52
8,including_boundaries,16,50331648,25165824,50.0,4.091588,53
9,including_boundaries,18,50331648,25165824,50.0,11.959154,57


In [12]:
serialization_summary = (
    pd.DataFrame([
        {
            'variant': variant,
            'compressed_model_path': (
                str(COMPRESSED_MODEL_PATHS[variant])
                if SAVE_COMPRESSED_MODEL
                else None
            ),
            'serialized_model_bytes': serialized_bytes,
        }
        for variant, serialized_bytes
        in serialized_model_bytes_by_variant.items()
    ])
    if RUN_FROM_SCRATCH
    else pd.DataFrame(
        loaded_artifact['results']['serialization']
    )
)

display(serialization_summary)

,variant,compressed_model_path,serialized_model_bytes
0,including_boundaries,None,None
1,excluding_boundaries,None,None


In [13]:
def json_records(frame):
    return json.loads(
        frame.to_json(orient='records', double_precision=15)
    )


if RUN_FROM_SCRATCH:
    artifact = {
        'schema_version': 1,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'environment': environment_record(),
        'configuration': {
            'model': {
                **json_value(model_config),
                'resolved_revision': getattr(
                    model.config,
                    '_commit_hash',
                    None,
                ),
            },
            'experiments': {
                variant: config.to_dict()
                for variant, config in experiment_configs.items()
            },
            'save_compressed_model': SAVE_COMPRESSED_MODEL,
            'compressed_model_paths': COMPRESSED_MODEL_PATHS,
        },
        'results': {
            'evaluations': json_records(evaluation_summary),
            'selections': json_records(selection_summary),
            'block_fitting': json_records(block_summary),
            'serialization': json_records(serialization_summary),
            'workflows': json_value(interleaved_results),
        },
    }

    ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    ARTIFACT_PATH.write_text(
        json.dumps(json_value(artifact), indent=2, allow_nan=False),
        encoding='utf-8',
    )
    print(f'Saved compression baseline artifact to {ARTIFACT_PATH}')
else:
    artifact = loaded_artifact
    print(f'Loaded compression baseline artifact from {ARTIFACT_PATH}')

Saved compression baseline artifact to /home/sgulas/thesis/development/data/results/model-compression-baselines/compression-baseline.json
